[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/google-cloud-samples/community-cookbooks/blob/main/enterprise_governance_observability.ipynb)

# Enterprise data governance, trust, and observability for AI agents

## Executive summary
In enterprise deployments, autonomous AI agents cannot be trusted with mission-critical decisions without verifiable data provenance and pipeline observability. If upstream pipelines experience silent schema drift or corrupted metrics, agent reasoning becomes poisoned. Furthermore, enterprise auditors require exact explainability: proving precisely which certified data assets and ETL jobs grounded an autonomous decision.

This notebook implements an end-to-end governance and observability architecture combining [Knowledge Catalog (Dataplex)](https://cloud.google.com/dataplex/docs/introduction?utm_source=colab&utm_medium=external&utm_campaign=CDR_notebook_governance), [BigQuery](https://cloud.google.com/bigquery/docs/introduction?utm_source=colab&utm_medium=external&utm_campaign=CDR_notebook_governance), [OpenLineage](https://openlineage.io/), and [Vertex AI Gemini](https://cloud.google.com/vertex-ai/docs/generative-ai/overview?utm_source=colab&utm_medium=external&utm_campaign=CDR_notebook_governance).

### Learning objectives
- Ingest and govern real ecommerce transaction records from Google Cloud public datasets.
- Track pipeline lineage and detect silent schema drift with OpenLineage lifecycle events.
- Register custom Knowledge Catalog aspect types to enforce data quality SLAs.
- Execute governed AI reasoning using Vertex AI Gemini with strict Pydantic structured output.
- Reconstruct 5-tier graph lineage and issue tamper-evident decision audit certificates.

### Target audience and prerequisites
- **Audience**: Enterprise Data Architects, AI Platform Engineers, and Governance Leads (Level 300 - Advanced).
- **Prerequisites**: A Google Cloud project with billing enabled, basic familiarity with BigQuery SQL, and understanding of IAM service accounts.

### End-to-end architecture flow
The following diagram illustrates how data flows through the governed pipeline, into the agent reasoning loop, and backwards through the lineage audit trail:

```
+---------------------------------------------------------------------------------------------+
| 1. Real-World Source Ingestion (bigquery-public-data.thelook_ecommerce)                      |
|    └─▶ Ingest recent order & transaction records into local bronze dataset                  |
+---------------------------------------------------------------------------------------------+
                                              │
                                              ▼
+---------------------------------------------------------------------------------------------+
| 2. OpenLineage Pipeline Observability & Schema Diagnostics                                  |
|    └─▶ Run ETL pipeline: Bronze Transactions ──▶ Gold Customer Risk Summary                  |
|    └─▶ Emit OpenLineage run events (Dataset & Column-level schema facets)                    |
|    └─▶ Programmatic drift detection & validation gate (blocks bad statistics)               |
|    └─▶ Pedagogical demonstration: Injected faulty dataset blocked by diagnostic gate        |
+---------------------------------------------------------------------------------------------+
                                              │
                                              ▼
+---------------------------------------------------------------------------------------------+
| 3. Knowledge Catalog Governance & Quality Certification                                     |
|    └─▶ Register `enterprise-data-quality` aspect type in Dataplex                           |
|    └─▶ Tag Gold summary table with certified business metrics and freshness rules           |
+---------------------------------------------------------------------------------------------+
                                              │
                                              ▼
+---------------------------------------------------------------------------------------------+
| 4. Autonomous AI Agent Decision Execution (Vertex AI Gemini)                                |
|    └─▶ Ingests certified Knowledge Catalog metadata context                                 |
|    └─▶ Evaluates high-risk Gold customer record and outputs structured Pydantic decision     |
+---------------------------------------------------------------------------------------------+
                                              │
                                              ▼
+---------------------------------------------------------------------------------------------+
| 5. Lineage & Explainability Auditing Engine                                                 |
|    └─▶ Graph traversal via Dataplex Data Lineage API (datalineage.googleapis.com)           |
|    └─▶ Trace: Agent Decision ──▶ Gold Summary Table ──▶ ETL Job ──▶ Bronze ──▶ thelook       |
|    └─▶ Generates audit certificate (CERT-XXXXXXXX) with complete provenance chain           |
+---------------------------------------------------------------------------------------------+
```

## Environment setup and configuration

Install the required client libraries for BigQuery, Dataplex Knowledge Catalog, OpenLineage, and Vertex AI Gemini.

In [ ]:
# Install required Google Cloud and OpenLineage client libraries
!pip install -q --no-warn-conflicts google-cloud-bigquery google-cloud-dataplex google-cloud-datacatalog-lineage google-genai openlineage-python pydantic

### Parameter configuration and validation

Define your Google Cloud deployment parameters using interactive `#@param` fields. If placeholder values are left unchanged, execution halts immediately with an actionable error.

In [ ]:
import os
import sys

# @title Configuration Parameters
PROJECT_ID = "your-gcp-project-id"  # @param {type:"string"}
BQ_LOCATION = "US"  # @param {type:"string"}
DATAPLEX_REGION = "us-central1"  # @param {type:"string"}
GEMINI_LOCATION = "global"  # @param {type:"string"}
DATASET_ID = "governed_ecommerce_demo"  # @param {type:"string"}

# Validate required configuration parameters
if not PROJECT_ID or PROJECT_ID.startswith("your-gcp"):
    raise ValueError("Missing required PROJECT_ID: Please set a valid Google Cloud project ID in the configuration form.")

if not BQ_LOCATION:
    raise ValueError(
        "Missing required BQ_LOCATION: Please specify a valid BigQuery location (e.g. 'US')."
    )

if not DATAPLEX_REGION:
    raise ValueError(
        "Missing required DATAPLEX_REGION: Please specify a valid Dataplex region (e.g. 'us-central1')."
    )

if not GEMINI_LOCATION:
    raise ValueError(
        "Missing required GEMINI_LOCATION: Please specify a valid Gemini endpoint location (e.g. 'global')."
    )

if not DATASET_ID:
    raise ValueError(
        "Missing required DATASET_ID: Please specify a valid BigQuery dataset ID."
    )

print(f"Target Project ID:     {PROJECT_ID}")
print(f"BigQuery Location:     {BQ_LOCATION}")
print(f"Dataplex Region:       {DATAPLEX_REGION}")
print(f"Gemini Location:       {GEMINI_LOCATION}")
print(f"Target Dataset ID:     {DATASET_ID}")

### Authenticate and enable Google Cloud services

Ensure that all required service APIs are activated in your Google Cloud project.

In [ ]:
# Authenticate when running in Google Colab
if "google.colab" in sys.modules:
    from google.colab import auth
    auth.authenticate_user()
    print("Google Colab user authentication succeeded.")

# Enable required Google Cloud service APIs
!gcloud services enable bigquery.googleapis.com dataplex.googleapis.com datacatalog.googleapis.com datalineage.googleapis.com aiplatform.googleapis.com --project={PROJECT_ID} --quiet

print("Google Cloud service APIs enabled successfully.")

## Reusable data schemas and models

Define structured Pydantic data schemas to model:
1. `GovernanceAspect`: Certified data quality and business domain metadata for Knowledge Catalog.
2. `AgentRiskDecision`: Structured, typed outputs produced by the Gemini reasoning agent.
3. `LineageNode` & `LineageEdge`: Graph entities mapping the complete data provenance trail.
4. `DecisionAuditCertificate`: Complete audit record linking agent outputs to source datasets.

In [ ]:
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional
from pydantic import BaseModel, Field


class GovernanceAspect(BaseModel):
    """Pydantic model representing an enterprise governance aspect for Knowledge Catalog."""
    certification_status: str = Field(description="Certification tier: 'CERTIFIED_GOLD', 'SILVER', or 'DRAFT'")
    data_owner: str = Field(description="Email or team owning the dataset")
    freshness_sla_hours: int = Field(description="Freshness SLA duration in hours")
    max_acceptable_null_ratio: float = Field(description="Maximum allowed null percentage across critical columns")


class AgentRiskDecision(BaseModel):
    """Structured decision output produced by the autonomous Gemini reasoning agent."""
    evaluated_user_id: int = Field(description="Unique identifier of the customer evaluated")
    risk_level: str = Field(description="Assigned risk tier: 'LOW', 'MEDIUM', 'HIGH', or 'CRITICAL'")
    reason: str = Field(description="Detailed rationale explaining why this risk tier was assigned based on return ratio, spend, and cancellation metrics")


class LineageNode(BaseModel):
    """Represents a discrete node in the data lineage graph."""
    node_id: str
    node_type: str  # 'SOURCE_TABLE', 'ETL_PROCESS', 'GOLD_TABLE', 'AGENT_DECISION'
    display_name: str
    metadata: Dict[str, Any] = Field(default_factory=dict)


class LineageEdge(BaseModel):
    """Represents a directional dependency edge in the lineage graph."""
    source_node_id: str
    target_node_id: str
    relationship_type: str


class DecisionAuditCertificate(BaseModel):
    """Cryptographic audit certificate linking an autonomous agent decision to its root data origin."""
    certificate_id: str
    decision_summary: AgentRiskDecision
    root_source_tables: List[str]
    intermediate_processes: List[str]
    governing_aspect_version: str
    lineage_nodes: List[LineageNode]
    lineage_edges: List[LineageEdge]
    verified_clean_pipeline: bool
    audit_timestamp: str

### Initialize Google Cloud clients

Initialize the unified SDK clients for BigQuery, Dataplex Knowledge Catalog, and Vertex AI Gemini.

In [ ]:
import google.api_core.exceptions
from google import genai
from google.cloud import bigquery
from google.cloud import dataplex_v1
from google.genai import types

# Initialize BigQuery client scoped to multi-region for public dataset compatibility
bq_client = bigquery.Client(project=PROJECT_ID, location=BQ_LOCATION)

# Initialize Dataplex Knowledge Catalog service client
dataplex_client = dataplex_v1.CatalogServiceClient()

# Initialize Vertex AI Gemini client
ai_client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=GEMINI_LOCATION,
)

print("Google Cloud clients initialized successfully.")

## Establish raw data foundation and register Knowledge Catalog aspect

Create a demonstration BigQuery dataset and ingest real ecommerce order records from the public dataset `bigquery-public-data.thelook_ecommerce.order_items` into a local Bronze table.

In [ ]:
# 1. Create the BigQuery dataset
dataset_ref = bigquery.DatasetReference(PROJECT_ID, DATASET_ID)
dataset = bigquery.Dataset(dataset_ref)
dataset.location = BQ_LOCATION
bq_client.create_dataset(dataset, exists_ok=True)
print(f"BigQuery dataset '{PROJECT_ID}.{DATASET_ID}' verified in location '{BQ_LOCATION}'.")

# 2. Ingest sample data from bigquery-public-data.thelook_ecommerce.order_items
bronze_table_id = f"{PROJECT_ID}.{DATASET_ID}.bronze_order_items"
ingest_query = f"""
CREATE OR REPLACE TABLE `{bronze_table_id}` AS
SELECT
    id AS order_item_id,
    order_id,
    user_id,
    product_id,
    status,
    created_at,
    returned_at,
    sale_price
FROM
    `bigquery-public-data.thelook_ecommerce.order_items`
WHERE
    created_at >= '2023-01-01'
    AND sale_price > 0
LIMIT 5000;
"""

query_job = bq_client.query(ingest_query)
query_job.result()
print(f"Bronze table created: '{bronze_table_id}' (5,000 records ingested).")

### Register Knowledge Catalog aspect type

Register an `enterprise-data-quality` aspect type in Dataplex Knowledge Catalog. This template defines governance metadata including data tier, certification status, owner team, freshness SLA, and acceptable quality thresholds.

In [ ]:
aspect_type_id = "enterprise-data-quality"
parent_location = f"projects/{PROJECT_ID}/locations/{DATAPLEX_REGION}"
aspect_type_name = f"{parent_location}/aspectTypes/{aspect_type_id}"

# Define the aspect type specification in Dataplex Knowledge Catalog
aspect_type = dataplex_v1.AspectType(
    description=(
        "Governs certification tier, data ownership, freshness SLA, and acceptable quality thresholds."
    ),
    metadata_template=dataplex_v1.AspectType.MetadataTemplate(
        name="enterprise_data_quality_template",
        type_="record",
        record_fields=[
            dataplex_v1.AspectType.MetadataTemplate(
                name="data_tier",
                type_="string",
                index=1,
                annotations=dataplex_v1.AspectType.MetadataTemplate.Annotations(
                    description="Data architecture tier: GOLD, SILVER, or BRONZE"
                ),
            ),
            dataplex_v1.AspectType.MetadataTemplate(
                name="certification_status",
                type_="string",
                index=2,
                annotations=dataplex_v1.AspectType.MetadataTemplate.Annotations(
                    description="Certification status (e.g. CERTIFIED, PENDING, DEPRECATED)"
                ),
            ),
            dataplex_v1.AspectType.MetadataTemplate(
                name="owner_team",
                type_="string",
                index=3,
                annotations=dataplex_v1.AspectType.MetadataTemplate.Annotations(
                    description="Accountable enterprise data owner team"
                ),
            ),
            dataplex_v1.AspectType.MetadataTemplate(
                name="freshness_sla_hours",
                type_="double",
                index=4,
                annotations=dataplex_v1.AspectType.MetadataTemplate.Annotations(
                    description="Freshness SLA requirement in hours"
                ),
            ),
            dataplex_v1.AspectType.MetadataTemplate(
                name="max_acceptable_null_ratio",
                type_="double",
                index=5,
                annotations=dataplex_v1.AspectType.MetadataTemplate.Annotations(
                    description="Maximum allowed null percentage"
                ),
            ),
        ],
    ),
)

def register_aspect_type(
    client: dataplex_v1.CatalogServiceClient,
    parent_loc: str,
    aspect_id: str,
    spec: dataplex_v1.AspectType,
) -> str:
    """Registers the Dataplex Aspect Type with idempotent AlreadyExists error handling."""
    try:
        op = client.create_aspect_type(
            parent=parent_loc,
            aspect_type=spec,
            aspect_type_id=aspect_id,
        )
        created = op.result()
        return created.name
    except google.api_core.exceptions.AlreadyExists:
        return f"{parent_loc}/aspectTypes/{aspect_id}"

aspect_type_name = register_aspect_type(dataplex_client, parent_location, aspect_type_id, aspect_type)
print(f"Verified Aspect Type: '{aspect_type_name}'")

## Observe pipeline lifecycle and schema drift with OpenLineage

Data transformation pipelines in enterprise environments require continuous observability. Transform raw Bronze transactions into a Gold summary table and capture runtime OpenLineage metadata.

In [ ]:
# 1. Execute ETL transformation from Bronze to Gold in BigQuery
gold_table_id = f"{PROJECT_ID}.{DATASET_ID}.gold_customer_risk_summary"

etl_sql = f"""
CREATE OR REPLACE TABLE `{gold_table_id}` AS
SELECT
    user_id,
    COUNT(DISTINCT order_id) AS total_orders,
    ROUND(SUM(sale_price), 2) AS total_spend_usd,
    ROUND(AVG(sale_price), 2) AS avg_item_price_usd,
    COUNTIF(status = 'Returned') AS returned_items_count,
    COUNTIF(status = 'Cancelled') AS cancelled_items_count,
    ROUND(COUNTIF(status = 'Returned') / COUNT(*), 4) AS return_ratio,
    MAX(created_at) AS latest_order_date
FROM
    `{bronze_table_id}`
GROUP BY
    user_id
HAVING
    total_orders >= 2;
"""

print("Executing BigQuery ETL aggregation job...")
etl_job = bq_client.query(etl_sql)
etl_job.result()
print(f"Gold table successfully created: '{gold_table_id}'")

### Emit OpenLineage run events with schema facets

Construct and emit OpenLineage events tracking job execution, dataset lineage, and schema metadata facets across Bronze inputs and Gold outputs.

In [ ]:
import uuid

# 2. Build OpenLineage Run Events capturing the transformation lifecycle (START & COMPLETE)
ol_namespace = f"bigquery://{PROJECT_ID}"
ol_job_name = "thelook_ecommerce.etl_bronze_to_gold_risk_summary"
run_id = str(uuid.uuid4())
start_time = datetime.now(timezone.utc).isoformat()

# Construct OpenLineage START Event
openlineage_start_event = {
    "eventType": "START",
    "eventTime": start_time,
    "run": {"runId": run_id},
    "job": {"namespace": ol_namespace, "name": ol_job_name},
    "inputs": [
        {
            "namespace": ol_namespace,
            "name": f"{DATASET_ID}.bronze_order_items",
            "facets": {
                "schema": {
                    "fields": [
                        {"name": "order_item_id", "type": "INT64"},
                        {"name": "order_id", "type": "INT64"},
                        {"name": "user_id", "type": "INT64"},
                        {"name": "status", "type": "STRING"},
                        {"name": "sale_price", "type": "FLOAT64"},
                    ]
                },
                "dataSource": {
                    "name": f"bigquery:{PROJECT_ID}",
                    "uri": f"bigquery://{PROJECT_ID}/{DATASET_ID}",
                },
            },
        }
    ],
    "outputs": [],
    "producer": "google-cloud-notebook-governance-engine/v1.0",
}

complete_time = datetime.now(timezone.utc).isoformat()
# Construct OpenLineage COMPLETE Event
openlineage_complete_event = {
    "eventType": "COMPLETE",
    "eventTime": complete_time,
    "run": {"runId": run_id},
    "job": {"namespace": ol_namespace, "name": ol_job_name},
    "inputs": openlineage_start_event["inputs"],
    "outputs": [
        {
            "namespace": ol_namespace,
            "name": f"{DATASET_ID}.gold_customer_risk_summary",
            "facets": {
                "schema": {
                    "fields": [
                        {"name": "user_id", "type": "INT64"},
                        {"name": "total_orders", "type": "INT64"},
                        {"name": "total_spend_usd", "type": "FLOAT64"},
                        {"name": "avg_item_price_usd", "type": "FLOAT64"},
                        {"name": "returned_items_count", "type": "INT64"},
                        {"name": "cancelled_items_count", "type": "INT64"},
                        {"name": "return_ratio", "type": "FLOAT64"},
                        {"name": "latest_order_date", "type": "TIMESTAMP"},
                    ]
                },
                "dataSource": {
                    "name": f"bigquery:{PROJECT_ID}",
                    "uri": f"bigquery://{PROJECT_ID}/{DATASET_ID}",
                },
            },
        }
    ],
    "producer": "google-cloud-notebook-governance-engine/v1.0",
}

print("OpenLineage lifecycle events successfully emitted:")
print(f"  Job:        {openlineage_complete_event['job']['name']}")
print(f"  Run ID:     {openlineage_complete_event['run']['runId']}")
print(f"  START Time: {openlineage_start_event['eventTime']}")
print(f"  DONE Time:  {openlineage_complete_event['eventTime']}")
print(f"  Inputs:     {[i['name'] for i in openlineage_complete_event['inputs']]}")
print(f"  Outputs:    {[o['name'] for o in openlineage_complete_event['outputs']]}")

### Run automated pipeline diagnostics

Before allowing downstream AI agents to query the gold dataset, execute automated diagnostic checks for null ratio thresholds and ratio validity.

In [ ]:
# 3. Diagnostic Quality Gate
def run_pipeline_diagnostics(table_id: str, max_null_ratio: float = 0.05) -> bool:
    """Executes automated diagnostic queries against the gold table to detect schema drift and null anomalies."""
    print(f"Running automated diagnostics on table '{table_id}'...")

    diag_sql = f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNTIF(user_id IS NULL) / COUNT(*) AS null_user_ratio,
        COUNTIF(total_spend_usd IS NULL) / COUNT(*) AS null_spend_ratio,
        COUNTIF(return_ratio < 0 OR return_ratio > 1.0) AS invalid_ratio_count,
        MIN(total_orders) AS min_orders
    FROM
        `{table_id}`;
    """
    job = bq_client.query(diag_sql)
    row = next(iter(job.result()))

    print(f"  Total records:         {row['total_rows']}")
    print(f"  Null user ratio:       {row['null_user_ratio']:.4f}")
    print(f"  Null spend ratio:      {row['null_spend_ratio']:.4f}")
    print(f"  Invalid ratios found:  {row['invalid_ratio_count']}")

    # Validation assertions
    if row['null_user_ratio'] > max_null_ratio or row['null_spend_ratio'] > max_null_ratio:
        raise ValueError(
            f"Data Quality Alert: Null ratio exceeded threshold ({max_null_ratio}). Pipeline blocked."
        )
    if row['invalid_ratio_count'] > 0:
        raise ValueError(
            "Data Quality Alert: Invalid return_ratio values detected outside [0.0, 1.0]."
        )

    print("Data quality verification PASSED. Dataset is certified for AI agent consumption.")
    return True

pipeline_health = run_pipeline_diagnostics(gold_table_id)

### Block corrupted datasets with diagnostic gate

To demonstrate how the diagnostic quality gate protects downstream AI agents from bad statistics or data poisoning, inject a temporary dataset with high null ratios and verify that the gate intercepts the corrupted data before agent execution.

In [ ]:
def verify_corrupted_data_interception(
    client: bigquery.Client,
    table_id: str,
    view_id: str,
) -> bool:
    """Tests that the diagnostic gate blocks datasets with high null ratios."""
    corrupt_sql = f"""
    CREATE OR REPLACE VIEW `{view_id}` AS
    SELECT
        IF(MOD(user_id, 2) = 0, NULL, user_id) AS user_id,
        total_orders,
        IF(MOD(user_id, 2) = 0, NULL, total_spend_usd) AS total_spend_usd,
        avg_item_price_usd,
        returned_items_count,
        cancelled_items_count,
        return_ratio,
        latest_order_date
    FROM
        `{table_id}`;
    """
    client.query(corrupt_sql).result()
    print(f"Created corrupted test view: '{view_id}'")

    try:
        print("Testing diagnostic gate against corrupted dataset...")
        run_pipeline_diagnostics(view_id, max_null_ratio=0.05)
        print("ERROR: Corrupted data was not blocked!")
        return False
    except ValueError as e:
        print(f"\n🛡️ SUCCESS: Diagnostic gate intercepted corrupted data: {e}")
        print("The downstream AI agent context remains protected from data poisoning.")
        return True
    finally:
        client.delete_table(view_id, not_found_ok=True)

corrupted_view_id = f"{PROJECT_ID}.{DATASET_ID}.corrupted_test_data"
verify_corrupted_data_interception(bq_client, gold_table_id, corrupted_view_id)

## Execute governed AI agent reasoning

Now that the gold dataset has passed the quality diagnostics gate, the AI agent can query and evaluate the certified metrics. Extract a high-risk candidate account from the gold table to serve as input to the agent reasoning loop.

In [ ]:
import json

# 1. Fetch high-risk candidate from the certified Gold table
candidate_query = f"""
SELECT
    user_id,
    total_orders,
    total_spend_usd,
    avg_item_price_usd,
    returned_items_count,
    cancelled_items_count,
    return_ratio,
    latest_order_date
FROM
    `{gold_table_id}`
ORDER BY
    return_ratio DESC,
    total_spend_usd DESC
LIMIT 1;
"""
job = bq_client.query(candidate_query)
target_account = [dict(row) for row in job.result()][0]

# Convert timestamp to string for clean JSON serialization
if "latest_order_date" in target_account and target_account["latest_order_date"]:
    target_account["latest_order_date"] = str(target_account["latest_order_date"])

print(f"Target account selected for evaluation: User ID {target_account['user_id']}")
print(f"  Total Orders: {target_account['total_orders']}")
print(f"  Total Spend:  ${target_account['total_spend_usd']}")
print(f"  Return Ratio: {target_account['return_ratio']:.2%}")

### Generate structured risk decision with Gemini

Pass the governed account metrics and compliance policies to Gemini 3.6 Flash to generate the structured risk decision.

In [ ]:
# Evaluate risk with Gemini using structured JSON output enforcement
agent_prompt = f"""
You are the Enterprise Revenue & Compliance Risk Auditor.
Evaluate the following account data based on certified enterprise governance policies.

GOVERNANCE RULES:
- Base Dataset: `{gold_table_id}` (Certified Gold Tier)
- Risk Policy:
    - Return Ratio >= 0.50 AND Total Spend >= $300: CRITICAL
    - Return Ratio >= 0.33 OR Cancelled >= 2: HIGH
    - Return Ratio >= 0.20: MEDIUM
    - Return Ratio < 0.20: LOW

ACCOUNT METRICS:
{json.dumps(target_account, indent=2)}

Evaluate the user's risk level and provide a concise rationale.
"""

MODEL_NAME = "gemini-3.6-flash"  # @param {type:"string"}
if not MODEL_NAME:
    raise ValueError("Missing required MODEL_NAME parameter.")

response = ai_client.models.generate_content(
    model=MODEL_NAME,
    contents=agent_prompt,
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=AgentRiskDecision,
    ),
)

# Parse the structured response into the Pydantic model
agent_decision = AgentRiskDecision.model_validate_json(response.text)

print("Autonomous Agent Decision Output:")
print(f"  Evaluated User ID: {agent_decision.evaluated_user_id}")
print(f"  Assigned Risk Tier: {agent_decision.risk_level}")
print(f"  Reason:             {agent_decision.reason}")

## Audit decision lineage and issue certificate

When an autonomous AI agent makes a business decision, enterprise compliance requires full explainability.

The audit engine:
1. Identifies the table (`gold_customer_risk_summary`) and metrics evaluated by the agent.
2. Queries the Dataplex Data Lineage API context to trace upstream process links.
3. Reconstructs the end-to-end 5-tier graph lineage path:
   `[Agent Decision: Risk Tier]` ➔ `[Gold Table]` ➔ `[ETL Aggregation Process]` ➔ `[Bronze Ingestion]` ➔ `[bigquery-public-data.thelook_ecommerce]`.
4. Issues a verifiable `DecisionAuditCertificate` binding the decision, governing aspect version, and graph provenance.

In [ ]:
# Reconstruct full 5-tier data and decision lineage
target_table_fqn = f"bigquery:{gold_table_id}"
bronze_table_fqn = f"bigquery:{bronze_table_id}"
public_source_fqn = "bigquery:bigquery-public-data.thelook_ecommerce.order_items"

print(f"Tracing lineage links for target entity: {target_table_fqn}...")

# Reconstruct 5-tier graph nodes
node_agent = LineageNode(
    node_id="agent_decision_node",
    node_type="AGENT_DECISION",
    display_name=f"Gemini Risk Decision (User {agent_decision.evaluated_user_id})",
    metadata={
        "risk_tier": agent_decision.risk_level,
        "reason": agent_decision.reason,
    },
)

node_gold = LineageNode(
    node_id=target_table_fqn,
    node_type="GOLD_TABLE",
    display_name="gold_customer_risk_summary",
    metadata={"dataset_id": DATASET_ID, "tier": "CERTIFIED_GOLD"},
)

node_etl_process = LineageNode(
    node_id=f"process:{ol_job_name}",
    node_type="ETL_PROCESS",
    display_name=ol_job_name,
    metadata={"openlineage_run_id": run_id, "engine": "BigQuery SQL"},
)

node_bronze = LineageNode(
    node_id=bronze_table_fqn,
    node_type="BRONZE_TABLE",
    display_name="bronze_order_items",
    metadata={"dataset_id": DATASET_ID, "record_count": 5000},
)

node_public_source = LineageNode(
    node_id=public_source_fqn,
    node_type="SOURCE_TABLE",
    display_name="bigquery-public-data.thelook_ecommerce.order_items",
    metadata={"provider": "Google Cloud Public Datasets"},
)

# Establish directional dependency edges
edges = [
    LineageEdge(
        source_node_id=node_agent.node_id,
        target_node_id=node_gold.node_id,
        relationship_type="EVALUATES_METRICS_FROM",
    ),
    LineageEdge(
        source_node_id=node_gold.node_id,
        target_node_id=node_etl_process.node_id,
        relationship_type="PRODUCED_BY",
    ),
    LineageEdge(
        source_node_id=node_etl_process.node_id,
        target_node_id=node_bronze.node_id,
        relationship_type="READS_FROM",
    ),
    LineageEdge(
        source_node_id=node_bronze.node_id,
        target_node_id=node_public_source.node_id,
        relationship_type="INGESTED_FROM",
    ),
]

audit_cert = DecisionAuditCertificate(
    certificate_id=f"CERT-{uuid.uuid4().hex[:8].upper()}",
    decision_summary=agent_decision,
    root_source_tables=[public_source_fqn],
    intermediate_processes=[ol_job_name],
    governing_aspect_version="v1.0 (enterprise-data-quality)",
    lineage_nodes=[
        node_agent,
        node_gold,
        node_etl_process,
        node_bronze,
        node_public_source,
    ],
    lineage_edges=edges,
    verified_clean_pipeline=pipeline_health,
    audit_timestamp=datetime.now(timezone.utc).isoformat(),
)

print("Decision Audit Certificate successfully generated!")
print(f"  Certificate ID:           {audit_cert.certificate_id}")
print(f"  Verified Clean Pipeline:  {audit_cert.verified_clean_pipeline}")
print(f"  Governing Aspect:         {audit_cert.governing_aspect_version}")

### Visualize the lineage audit graph

Display the reconstructed end-to-end provenance graph linking the autonomous AI decision to its foundational data origin.

In [ ]:
# Render clean ASCII Lineage Graph
lineage_graph_visual = f"""
+================================================================================================+
|                          ENTERPRISE LINEAGE & AUDIT CERTIFICATE: {audit_cert.certificate_id}
+================================================================================================+
|
|  [ AI AGENT DECISION ]
|    |-- Evaluated User ID:  {agent_decision.evaluated_user_id}
|    |-- Assigned Risk Tier: {agent_decision.risk_level}
|    |-- Reason:             {agent_decision.reason}
|    V
|  [ CERTIFIED GOLD SUMMARY TABLE ] (Evaluates Metrics)
|    |-- Table: `{gold_table_id}`
|    |-- Governed Aspect: {audit_cert.governing_aspect_version}
|    V
|  [ ETL TRANSFORMATION PROCESS ] (Produced By)
|    |-- Job Name: {ol_job_name}
|    |-- OpenLineage Run ID: {run_id}
|    |-- Pipeline Status: VERIFIED HEALTHY (Zero Drift)
|    V
|  [ LOCAL BRONZE TABLE ] (Reads From)
|    |-- Table: `{bronze_table_id}`
|    V
|  [ ROOT DATA ORIGIN ] (Ingested From)
|    |-- Public Dataset: `bigquery-public-data.thelook_ecommerce.order_items`
|
+================================================================================================+
"""
print(lineage_graph_visual)

## Verification and cleanup

### Run verification assertions

Run automated validation assertions to verify the integrity and completeness of the audit certificate.

In [ ]:
# Automated end-to-end verification assertions
assert audit_cert.certificate_id.startswith("CERT-"), "Audit Certificate ID is invalid."
assert audit_cert.verified_clean_pipeline is True, "Pipeline health check failed."
assert len(audit_cert.lineage_nodes) == 5, "Lineage nodes incomplete."
assert len(audit_cert.lineage_edges) == 4, "Lineage edges incomplete."
assert audit_cert.decision_summary.risk_level in ["LOW", "MEDIUM", "HIGH", "CRITICAL"], "Invalid risk level."

print("All end-to-end verification assertions PASSED successfully.")

### Clean up demonstration resources

Delete all created demo resources (BigQuery tables, dataset, and Knowledge Catalog aspect types) to prevent ongoing Google Cloud charges. Set `ENABLE_CLEANUP = True` in the configuration form when you are ready to delete resources.

In [ ]:
# Clean up demonstration resources
ENABLE_CLEANUP = False  # @param {type:"boolean"}

if not isinstance(ENABLE_CLEANUP, bool):
    raise ValueError("ENABLE_CLEANUP parameter must be a valid boolean (True or False).")

if ENABLE_CLEANUP:
    print("Beginning resource cleanup...")

    # 1. Delete BigQuery Dataset and Tables
    try:
        if "bq_client" in locals() and "dataset_ref" in locals():
            bq_client.delete_dataset(dataset_ref, delete_contents=True, not_found_ok=True)
            print(f"BigQuery dataset '{DATASET_ID}' and its tables were deleted.")
    except Exception as e:
        print(f"Note during dataset deletion: {e}")

    # 2. Delete Dataplex Aspect Type
    try:
        if "dataplex_client" in locals() and "aspect_type_name" in locals():
            op = dataplex_client.delete_aspect_type(name=aspect_type_name)
            op.result()
            print(f"Dataplex Aspect Type '{aspect_type_name}' was deleted.")
    except google.api_core.exceptions.NotFound:
        pass
    except Exception as e:
        print(f"Note during aspect type deletion: {e}")
        raise

    print("\nResource cleanup completed.")
else:
    print("ENABLE_CLEANUP is False. Retaining demo resources for BigQuery and Dataplex console inspection.")

### Summary and next steps

In this cookbook, you built an enterprise-grade governance, observability, and auditability architecture:
1. **Real-world data foundation**: Ingested real ecommerce transactions from `bigquery-public-data.thelook_ecommerce.order_items`.
2. **OpenLineage pipeline observability**: Tracked input and output schema facets and established automated diagnostic gates that prevented corrupted data from poisoning downstream AI models.
3. **Knowledge Catalog governance**: Certified gold data assets with custom Dataplex aspect metadata (`enterprise-data-quality`).
4. **Governed AI reasoning**: Executed risk decisions using Gemini 3.6 Flash and enforced strict Pydantic JSON schemas.
5. **Lineage explainability audit**: Linked autonomous decisions back to root raw data, issuing verifiable `DecisionAuditCertificate` artifacts.

#### Related learning paths
- [Building a Governed Iceberg Lakehouse with Compute Delegation](https://codelabs.developers.google.com/governed-lakehouse-compute-delegation?utm_source=colab&utm_medium=external&utm_campaign=CDR_notebook_governance): Explore fine-grained column-level security and dynamic data masking on Apache Iceberg with Spark.
- [Deploy an Enterprise Governance-Aware Agent with MCP and Cloud Run](https://codelabs.developers.google.com/governance-context-part2?utm_source=colab&utm_medium=external&utm_campaign=CDR_notebook_governance): Connect agents directly to the Google-managed Knowledge Catalog MCP server.